# Calibration Data Preparation Protocol
*(Pre-Adaptation Phase of AURA)*

## Quick Start: Install Dependencies
Run the following cell to ensure you have the required libraries installed in your current kernel.


In [ ]:
%pip install pandas numpy


## Objective
Construct a **derived, analysis-ready calibration dataset** by temporally aligning raw telemetry windows with discrete death events. This dataset serves to characterise baseline behavioural tendencies and compare gameplay modes under controlled conditions.

## Guiding Principles
1.  **Raw telemetry is immutable**: Original logs remain changed.
2.  **Calibration data is derived**: No retroactive smoothing.
3.  **Temporal alignment**: Deaths are mapped to the time window they occurred in.
4.  **Reproducibility**: This notebook can be re-run on new data.


In [ ]:
import pandas as pd
import numpy as np
import os

# Define Paths
DATA_DIR = 'data'
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

TELEMETRY_FILE = os.path.join(DATA_DIR, 'telemetry.calibration_telemetry.csv')
DEATHS_FILE = os.path.join(DATA_DIR, 'telemetry.deathevents.csv')
USERS_FILE = os.path.join(DATA_DIR, 'telemetry.users.csv')
OUTPUT_FILE = os.path.join(PROCESSED_DIR, 'calibration_dataset.csv')

print(f"Reading from: {DATA_DIR}")
print(f"Writing to: {PROCESSED_DIR}")


## Step 1: Load Data
We load both the continuous telemetry windows (30s intervals) and the discrete death events. We explicitly parse timestamps to ensure accurate alignment.


In [ ]:
# Load Datasets
df_telemetry = pd.read_csv(TELEMETRY_FILE)
df_deaths = pd.read_csv(DEATHS_FILE)

# Robust Column Detection (Handle 'time' vs 'timestamp')
def get_time_col(df, name="dataset"):
    if 'time' in df.columns:
        return 'time'
    elif 'timestamp' in df.columns:
        return 'timestamp'
    else:
        raise KeyError(f"Neither 'time' nor 'timestamp' column found in {name}. Columns: {df.columns.tolist()}")

tel_time_col = get_time_col(df_telemetry, "Telemetry")
death_time_col = get_time_col(df_deaths, "Deaths")

print(f"Using '{tel_time_col}' for Telemetry and '{death_time_col}' for Deaths.")

# Ensure sorting and consistent types
# Convert timestamps to datetime objects for accurate comparison
df_telemetry['time_dt'] = pd.to_datetime(df_telemetry[tel_time_col])
df_deaths['time_dt'] = pd.to_datetime(df_deaths[death_time_col])

df_telemetry = df_telemetry.sort_values(by=['userId', 'time_dt']).reset_index(drop=True)
df_deaths = df_deaths.sort_values(by=['userId', 'time_dt']).reset_index(drop=True)

print(f"Telemetry Rows: {len(df_telemetry)}")
print(f"Death Events: {len(df_deaths)}")


## Step 1.5: User Validation (Optional)
We check if the `userId`s in the Death events actually exist in our Users registry.


In [ ]:
if os.path.exists(USERS_FILE):
    df_users = pd.read_csv(USERS_FILE)
    print(f"Loaded {len(df_users)} users.")
    
    # Standardize ID column (assuming '_id' in users matches 'userId' in deaths)
    user_ids = set(df_users['_id'].astype(str) if '_id' in df_users.columns else df_users['userId'].astype(str))
    death_user_ids = set(df_deaths['userId'].astype(str))
    
    # Check for mismatches
    missing_users = death_user_ids - user_ids
    
    if len(missing_users) > 0:
        print(f"WARNING: Found {len(missing_users)} User IDs in Death Events that are NOT in Users file:")
        # Print ALL missing user IDs as requested
        for uid in missing_users:
            print(f" - Missing User ID: {uid}")
    else:
        print("Validation Passed: All death event User IDs exist in the Users file.")
else:
    print("Users file not found. Skipping validation.")


## Step 2: Temporal Alignment Rule
**The Rule:** A death event is associated with the **telemetry window whose timestamp range contains the death timestamp**.

Since our telemetry timestamps represent the *window boundary* (likely the end of the window or the sampling point), we align each death to the **nearest following telemetry timestamp** within the same User and Mode.

*   `Death Time <= Telemetry Time`
*   Match on `userId` (and `modeId` if available)

We use `pd.merge_asof` with `direction='forward'` to find the first telemetry timestamp that is greater than or equal to the death timestamp.


In [ ]:
# 1. Prepare Telemetry for lookup
telemetry_lookup = df_telemetry[['userId', 'time_dt']].copy()
# Explicitly use datetime column for matching
telemetry_lookup['window_time_key'] = telemetry_lookup['time_dt']

if 'modeId' in df_telemetry.columns:
    telemetry_lookup['modeId'] = df_telemetry['modeId']

# 2. Match
by_columns = ['userId']
if 'modeId' in df_deaths.columns and 'modeId' in df_telemetry.columns:
    by_columns.append('modeId')

# merge_asof requires sorting on the matching key
aligned = pd.merge_asof(
    df_deaths.sort_values('time_dt'),
    telemetry_lookup.sort_values('time_dt'),
    on='time_dt',
    by=by_columns,
    direction='forward',
    allow_exact_matches=True
)

print(f"Deaths matched to windows: {aligned['window_time_key'].notna().sum()} / {len(aligned)}")


## Step 3: Analysis of Unmatched Deaths
Here we isolate exactly **which** death events failed to match and **why** (e.g. invalid user, or death happened after game session ended).


In [ ]:
# Filter for unmatched deaths
unmatched_deaths = aligned[aligned['window_time_key'].isna()]

print(f"Total Unmatched Deaths: {len(unmatched_deaths)}")

if len(unmatched_deaths) > 0:
    print("\n--- Detailed Failure Report ---")
    # Get list of valid telemetry users for reference
    valid_telemetry_users = set(df_telemetry['userId'].unique())
    
    for idx, row in unmatched_deaths.iterrows():
        uid = row['userId']
        dt = row['time_dt']
        
        reason = "Unknown"
        if uid not in valid_telemetry_users:
            reason = "User ID not found in Telemetry data"
        else:
            # Check if it was a time issue (happened after last window)
            user_windows = df_telemetry[df_telemetry['userId'] == uid]
            last_window = user_windows['time_dt'].max()
            if dt > last_window:
                reason = f"Temporal Mismatch: Death ({dt}) happened AFTER the last recorded window ({last_window})"
            else:
                reason = "Temporal/Mode Mismatch (No valid following window found)"
                
        print(f"Death Event {idx}: User={uid}, Time={dt} -> FAILED: {reason}")


## Step 4: Aggregation & Selection
We count how many deaths correspond to each window. Then we finalize the dataset.


In [ ]:
# 3. Aggregate Deaths
aligned_clean = aligned[aligned['window_time_key'].notna()]

# Group by User and Window Time to count deaths
death_counts = aligned_clean.groupby(['userId', 'window_time_key']).size().reset_index(name='deathCountInWindow')
death_counts['deathOccurredInWindow'] = 1

# 4. Merge back into main Telemetry dataset
# matching on userId and time_dt
df_calibration = pd.merge(
    df_telemetry,
    death_counts,
    left_on=['userId', 'time_dt'],
    right_on=['userId', 'window_time_key'],
    how='left'
)

# 5. Fill NaNs
df_calibration['deathCountInWindow'] = df_calibration['deathCountInWindow'].fillna(0).astype(int)
df_calibration['deathOccurredInWindow'] = df_calibration['deathOccurredInWindow'].fillna(0).astype(int)

# Cleanup temporary columns
drop_cols = ['window_time_key', 'time_dt']
df_calibration.drop(columns=[c for c in drop_cols if c in df_calibration.columns], inplace=True)


## Step 5: Save Output & Data Verification
We save the file to `data/processed/calibration_dataset.csv`.

### Metric Naming Cleanup
We also strip the `metrics.` prefix from column names to match the configuration file.


In [ ]:
# Rename columns: metrics.enemiesHit -> enemiesHit
df_calibration.rename(columns=lambda x: x.replace('metrics.', ''), inplace=True)

# Save
df_calibration.to_csv(OUTPUT_FILE, index=False)
print(f"Saved calibration dataset to: {OUTPUT_FILE}")

# Print Column Types for Verification
print("\n--- Final Dataset Schema ---")
print(df_calibration.dtypes)
